In [ ]:
import sys

# Replace this string with the exact 'Location' path from your command prompt!
global_package_path = r"C:\Users\santo\AppData\Local\Programs\Python\Python312\Lib\site-packages"

# Add it to Jupyter's system path if it isn't there already
if global_package_path not in sys.path:
    sys.path.append(global_package_path)

print("Path added! Let's try importing again...")

# Now try the import!
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration
print("✅ Transformers imported successfully!")

In [3]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("✅ Transformers imported successfully!")

c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'is_opaque_value' from 'torch._library.opaque_object' (c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\_library\opaque_object.py)

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("🧠 Configuring 4-Bit Quantization...")

# 1. Setup 4-bit configuration to compress the model and save VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "llava-hf/llava-1.5-7b-hf"

print(f"📥 Downloading and Loading Model: {model_id}")
print("⏳ This will take a few minutes depending on your internet speed...")

# 2. Load the Processor (this translates your images and text into AI numbers)
processor = AutoProcessor.from_pretrained(model_id)

# 3. Load the base AI model in 4-bit mode directly to your RTX 3050
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" 
)

print("⚙️ Preparing Model for LoRA Fine-Tuning...")

# 4. Enable gradient checkpointing to prevent Out-of-Memory (OOM) errors
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 5. Setup the LoRA Configuration (the small trainable adapter)
lora_config = LoraConfig(
    r=16, # Rank of the adapter
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Target attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 6. Attach the LoRA adapter to the base model
model = get_peft_model(model, lora_config)

print("\n✅ Model successfully loaded and ready for training!")
# This will print out exactly how many parameters we are training vs freezing
model.print_trainable_parameters()

In [ ]:
import sys

print("YOUR JUPYTER IS USING THIS PYTHON:")
print(sys.executable)
print("-" * 50)
print("COPY AND PASTE THIS ENTIRE LINE INTO YOUR WINDOWS COMMAND PROMPT:")
print(f'"{sys.executable}" -m pip install transformers datasets peft accelerate bitsandbytes torch torchvision regex --force-reinstall')

YOUR JUPYTER IS USING THIS PYTHON:
c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\python.exe
--------------------------------------------------
COPY AND PASTE THIS ENTIRE LINE INTO YOUR WINDOWS COMMAND PROMPT:
"c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\python.exe" -m pip install transformers datasets peft accelerate bitsandbytes torch torchvision regex --force-reinstall


I am working on a group project of building a healthcare AI system which will take inputs (in the form of text and images of radiology and doctor's prescription) and then diagnose the diseases and prescribe treatments and medicines based on the inputs but with a warning of consulting a doctor. These are the files with which I have started the work. I want to use fine-tuning instead of building the AI model from scratch. 10 of the total files I have attached with my prompt, I will send you the rest files in my next prompts. Subsequently, you just have to tell me what to do next step-by-step. Please tell me one step at once not all and only move to the next step when prompted.

Are you ready to read the rest files??



In [ ]:
import torch

def check_device():
    # Check if CUDA is available
    if torch.cuda.is_available():
        print("✅ CUDA is available. GPU will be used.")
        device = torch.device("cuda")
    else:
        print("❌ CUDA is NOT available. CPU will be used.")
        device = torch.device("cpu")
    
    # Create a sample tensor on the selected device
    tensor = torch.zeros((2, 3), device=device)
    
    # Display device info
    print(f"Tensor is on device: {tensor.device}")
    
    # If GPU is available, show GPU name
    if device.type == "cuda":
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0)} bytes")
        print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0)} bytes")

if __name__ == "__main__":
    check_device()


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llava-finetuned",
    
    # 1. FIX THE VRAM OVERFLOW
    per_device_train_batch_size=1, # MUST be 1 for a 6GB card
    gradient_accumulation_steps=16, # Increased to maintain effective batch size
    
    # 2. FIX THE WINDOWS DEADLOCK
    dataloader_num_workers=0, # MUST be 0 on Windows Jupyter to prevent hanging
    
    # 3. MEMORY OPTIMIZATIONS
    fp16=True, 
    optim="paged_adamw_8bit", # Great choice, keep this!
    gradient_checkpointing=True, # Crucial for 6GB VRAM
    
    # 4. REDUCE INITIAL SCOPE FOR TESTING
    max_steps=100, # Change from 20268 to 100 just to make sure it works first!
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    remove_unused_columns=False, 
    report_to="none"
)

print("1. Loading Harmonized Dataset...")
# Load the master dataset we just created
dataset = load_dataset("json", data_files="Final_Multimodal_Dataset.json", split="train")
print(f"Loaded {len(dataset)} records.")

print("\n2. Configuring 4-bit Quantization (Crucial for 6GB VRAM)...")
# This shrinks the model's memory footprint so it fits on your RTX 3050
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# We will use the standard open-source LLaVA model
model_id = "llava-hf/llava-1.5-7b-hf" 
print(f"Loading Base Model: {model_id}. This might take a few minutes to download...")

# Load processor (handles both text and images)
processor = AutoProcessor.from_pretrained(model_id)

# Load the AI model in compressed 4-bit mode
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto" # Automatically places it on your GPU
)

print("\n3. Preparing for QLoRA Fine-Tuning...")
# Prepare model for memory-efficient training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False # Required to save VRAM during training

# Set up LoRA Configuration (The tiny trainable adapters)
lora_config = LoraConfig(
    r=8, # Rank - kept low to save memory
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Target specific attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the adapters to the model
model = get_peft_model(model, lora_config)

# Print memory footprint and trainable parameters
model.print_trainable_parameters()
print("\n✅ Model Successfully Loaded and Ready for 6GB VRAM Training!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


1. Loading Harmonized Dataset...
Loaded 279214 records.

2. Configuring 4-bit Quantization (Crucial for 6GB VRAM)...
Loading Base Model: llava-hf/llava-1.5-7b-hf. This might take a few minutes to download...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [1]:
from transformers import Trainer
from PIL import Image
import numpy as np

print("\n4. Truncating Dataset for Testing...")
# WE MUST TEST ON A TINY SUBSET FIRST! 
# Preprocessing 280k records will freeze a 16GB RAM PC. Let's test on 10 records.
small_dataset = dataset.select(range(10)) 

print("\n5. Creating Custom Data Collator for LLaVA...")
def collate_fn(examples):
    texts = []
    images = []
    
    for example in examples:
        # Format the conversation specifically for LLaVA
        conv = example['conversations']
        prompt = f"USER: {conv[0]['value']}\nASSISTANT: {conv[1]['value']}"
        texts.append(prompt)
        
        # Safely load image if it exists, otherwise create a blank dummy image
        if 'image' in example and example['image']:
            try:
                images.append(Image.open(example['image']).convert("RGB"))
            except:
                images.append(Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))
        else:
            images.append(Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))

    # Process everything with strict max_length to prevent VRAM explosions
    batch = processor(
        text=texts, 
        images=images, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=256 # Strictly limited for 6GB VRAM
    )
    
    # Create the labels needed for training
    batch["labels"] = batch["input_ids"].clone()
    
    return batch

print("\n6. Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_dataset,
    data_collator=collate_fn,
)

print("\n7. Starting Training (Testing Memory Limits!)...")
# If this hangs, open Windows Task Manager -> Performance -> GPU
# Check if "Shared GPU memory" is filling up.
trainer.train()

c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



4. Truncating Dataset for Testing...


NameError: name 'dataset' is not defined

In [ ]:
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

# Replace the string below with the token you just copied
login("HF_TOKEN")

print("1. Configuring Training Arguments for 6GB VRAM & Windows...")
training_args = TrainingArguments(
    output_dir="./llava-finetuned",
    per_device_train_batch_size=1,      
    gradient_accumulation_steps=16,     
    dataloader_num_workers=0,           
    fp16=True, 
    # --- CHANGED THIS LINE BELOW ---
    optim="adamw_8bit",                 # Removed "paged_" to prevent the Windows hard crash
    gradient_checkpointing=True,        
    max_steps=100,                      
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    remove_unused_columns=False, 
    report_to="none"
)

print("2. Loading Harmonized Dataset...")
dataset = load_dataset("json", data_files="Final_Multimodal_Dataset.json", split="train")
print(f"Loaded {len(dataset)} records.")

print("3. Truncating Dataset for Testing...")
# Test on just 10 records so we don't freeze your 16GB of system RAM!
small_dataset = dataset.select(range(10))

print("4. Configuring 4-bit Quantization...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model_id = "llava-hf/llava-1.5-7b-hf" 
print(f"5. Loading Base Model: {model_id} (This takes a moment)...")
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map={"": 0}  # <--- CHANGED THIS FROM "auto"
)

print("6. Preparing QLoRA Adapters...")
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False 

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

print("7. Setting up Data Collator...")
def collate_fn(examples):
    texts = []
    images = []
    
    for example in examples:
        conv = example['conversations']
        # Clean up tags and format specifically for LLaVA
        human_text = conv[0]['value'].replace('<image>', '').strip()
        prompt = f"USER: <image>\n{human_text}\nASSISTANT: {conv[1]['value']}"
        texts.append(prompt)
        
        # Load image or provide a blank dummy image if it's a text-only medical question
        if 'image' in example and example['image']:
            try:
                images.append(Image.open(example['image']).convert("RGB"))
            except:
                images.append(Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))
        else:
            images.append(Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))

    # Process with an adjusted max_length to fit the 576 image tokens!
    batch = processor(
        text=texts, 
        images=images, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=768 # <--- INCREASED TO 768
    )
    batch["labels"] = batch["input_ids"].clone()
    return batch

print("8. Initializing Trainer & Starting Training...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_dataset,
    data_collator=collate_fn,
)

# --- ADD THIS LINE TO CLEAR VRAM FRAGMENTATION ---
torch.cuda.empty_cache()

# Start the actual training process!
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


1. Configuring Training Arguments for 6GB VRAM & Windows...
2. Loading Harmonized Dataset...
Loaded 279214 records.
3. Truncating Dataset for Testing...
4. Configuring 4-bit Quantization...
5. Loading Base Model: llava-hf/llava-1.5-7b-hf (This takes a moment)...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

6. Preparing QLoRA Adapters...
7. Setting up Data Collator...
8. Initializing Trainer & Starting Training...


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,9.564552
20,6.523241
30,4.131997
40,3.874598
50,3.766071
60,3.671033
70,3.580478
80,3.515972
90,3.480493
100,3.468254


TrainOutput(global_step=100, training_loss=4.557669067382813, metrics={'train_runtime': 4706.2712, 'train_samples_per_second': 0.34, 'train_steps_per_second': 0.021, 'total_flos': 3.16803043172352e+16, 'train_loss': 4.557669067382813, 'epoch': 100.0})